# 02 · Attempt Propensity π(X)

**Purpose:** Estimate P(FG Attempt | game state X) via rolling-origin multinomial
logistic regression over actions (FG, Punt, Go-for-it). Produces inverse-probability
weights (IPW) used by notebook 03 to correct for selection bias in the outcome model.

**Inputs:**
- `data/fg_all.csv` (from notebook 01)

**Outputs:**
- `reports/attempt_pi/attempt_pi_oof_predictions_final.csv`
- `reports/attempt_pi/attempt_pi_metrics_by_season.csv`

**IPW Weight Pipeline (5 steps):**
1. Clip propensities to [0.02, 0.98]
2. Stabilized IPW: w_raw = prevalence / p_clipped
3. Hájek season-normalize (mean weight = 1 per season)
4. 3σ cap (remove extreme outliers)
5. Re-normalize (mean = 1 again)

In [1]:
# ============================================================
# 1. Parameters
# ============================================================
PROJECT_ROOT <- sub('[/\\\\][^/\\\\]*$', '', getwd())

data_dir    <- file.path(PROJECT_ROOT, 'data')
reports_dir <- file.path(PROJECT_ROOT, 'reports')
reports_pi_dir <- file.path(reports_dir, 'attempt_pi')

# Rolling-origin window size (train on prior N seasons)
WINDOW        <- 3L
# First season to generate OOF predictions for
OOF_START     <- 2015L
# Probability clipping bounds
CLIP_MIN      <- 0.02
CLIP_MAX      <- 0.98
# Field position spline basis. Deliberately NOT matched to notebook 03's distance
# basis. The coaching decision is defined over the whole field, the kick outcome
# only over kickable distances, so the two models resolve position on different
# supports by design. Encoding the decision on a hypothetical kick distance
# (yardline_100 + 17) forced a 70-yard truncation that hid every punt taken from
# outside kicking range - i.e. most of them - from the model of the decision.
FP_KNOTS      <- c(15, 30, 45, 60, 75)
FP_BOUNDS     <- c(1, 100)
# yardline_100 <= 53 is a 70-yard attempt: the kickable subset. IPW weights are
# normalized over this subset so M3's estimand stays "plausible kicks" even though
# the decision model itself is fitted across the full field.
IN_RANGE_YL   <- 53L

set.seed(20240517)
if (!dir.exists(reports_pi_dir)) dir.create(reports_pi_dir, recursive = TRUE)
message('PROJECT_ROOT: ', PROJECT_ROOT)

PROJECT_ROOT: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model



In [2]:
# ============================================================
# 2. Imports
# ============================================================
dependencies <- c(
  'dplyr', 'tibble', 'tidyr', 'readr', 'stringr', 'purrr', 'ggplot2',
  'splines', 'splines2', 'pROC', 'nnet'
)
installed <- rownames(installed.packages())
for (pkg in dependencies) {
  if (!pkg %in% installed) install.packages(pkg)
  suppressPackageStartupMessages(library(pkg, character.only = TRUE))
}
message('Libraries loaded.')

Libraries loaded.



In [3]:
# ============================================================
# 3. Helper Functions
# ============================================================

# Natural cubic spline basis on field position (yardline_100, yards from the
# opponent end zone). Natural splines are linear beyond the boundary knots, so
# unlike a B-spline they cannot produce runaway basis values at the extremes.
# With boundary knots at 1 and 100 every possible field position is inside the
# support, so nothing is ever extrapolated.
build_field_position_ns <- function(x, knots = FP_KNOTS, bknots = FP_BOUNDS) {
  b <- splines::ns(x, knots = knots, Boundary.knots = bknots)
  b <- as.data.frame(b)
  names(b) <- paste0('fp_', seq_len(ncol(b)))
  b
}

# Effective Sample Size
ess <- function(w) {
  w <- w[is.finite(w)]
  if (!length(w)) return(NA_real_)
  (sum(w)^2) / sum(w^2)
}

brier   <- function(y, p) mean((y - p)^2, na.rm = TRUE)
logloss <- function(y, p, eps = 1e-15)
  -mean(y * log(pmin(pmax(p, eps), 1-eps)) + (1-y) * log(1 - pmin(pmax(p, eps), 1-eps)),
        na.rm = TRUE)

## 4. Load Data & Build Decision Frame

In [4]:
fg_path <- file.path(data_dir, 'fg_all.csv')
stopifnot(file.exists(fg_path))
fg_all <- readr::read_csv(fg_path, show_col_types = FALSE)
message('fg_all loaded: ', nrow(fg_all), ' rows | seasons: ',
        min(fg_all$season), '-', max(fg_all$season))

fg_all loaded: 137668 rows | seasons: 2000-2025



In [5]:
# Build 4th-down decision frame (exclude PATs)
# action levels: go_for_it (reference), punt, FG
decision <- fg_all %>%
  filter(
    is_pat == 0L,
    play_type_original %in% c('field_goal', 'pass', 'run', 'punt'),
    !is.na(yardline_100),
    !is.na(game_seconds_remaining)
  ) %>%
  mutate(
    in_range   = as.integer(yardline_100 <= IN_RANGE_YL),
    attempt_fg = as.integer(play_type_original == 'field_goal'),
    action = dplyr::case_when(
      play_type_original == 'field_goal' ~ 'FG',
      play_type_original == 'punt'       ~ 'punt',
      TRUE                               ~ 'go_for_it'
    ),
    # Score state flags
    go_ahead       = as.integer(!is.na(score_differential) & score_differential >= -2 & score_differential <= 0),
    one_score_down = as.integer(!is.na(score_differential) & score_differential <= -4 & score_differential >= -8),
    to_tie         = as.integer(score_differential == -3),
    extend_lead    = as.integer(!is.na(score_differential) & score_differential >= 1),
    two_score_down = as.integer(!is.na(score_differential) & score_differential <= -9 & score_differential >= -11),
    # Venue/weather (coerce NAs to 0 for sparse flags)
    indoors      = dplyr::coalesce(as.integer(indoors),      0L),
    is_turf      = dplyr::coalesce(as.integer(is_turf),      0L),
    high_altitude = dplyr::coalesce(as.integer(high_altitude), 0L)
  )

# qb_kneel / qb_spike are not fourth-down decisions (victory formation, clock
# kills). They are excluded by the play_type_original filter above; assert it.
stopifnot(!any(decision$play_type_original %in% c('qb_kneel', 'qb_spike')))
stopifnot(all(decision$yardline_100 >= 1 & decision$yardline_100 <= 100))

message('Decision frame: ', nrow(decision), ' plays | ',
        sum(decision$attempt_fg), ' FG, ',
        sum(decision$action == 'punt'), ' punt, ',
        sum(decision$action == 'go_for_it'), ' go')
message('  field position spans yardline_100 ',
        min(decision$yardline_100), '-', max(decision$yardline_100),
        ' | in kicking range (<= ', IN_RANGE_YL, '): ', sum(decision$in_range),
        ' (', round(100 * mean(decision$in_range), 1), '%)')

Decision frame: 105249 plays | 26766 FG, 63451 punt, 15032 go



  field position spans yardline_100 1-99 | in kicking range (<= 53): 53757 (51.1%)



In [6]:
# ============================================================
# 5. Feature Construction
# ============================================================

# Natural spline basis on field position (full field, 1-100)
fp_X <- build_field_position_ns(decision$yardline_100)

# Game clock. A log transform rather than a natural spline: the effect of the
# clock on coaching urgency is monotone and concave (a minute matters far more
# with two minutes left than with twenty), which log1p captures in one column.
# It also removes the last leakage in this notebook - splines::ns() derived its
# knots from the pooled dataset and those columns were then used inside the
# rolling-origin folds, so held-out seasons informed the basis definition. A
# fixed transform has no estimated parameters and cannot leak. This mirrors how
# quarter_seconds_remaining is already handled two lines below.
log_game_time <- log1p(pmax(as.numeric(decision$game_seconds_remaining), 0))

# Quarter-time interaction features
quarter_sec  <- as.numeric(decision$quarter_seconds_remaining)
log_qtr_time <- log1p(pmax(quarter_sec, 0))
q2_flag      <- as.integer(as.integer(decision$qtr) == 2)
q4_flag      <- as.integer(as.integer(decision$qtr) == 4)

X <- cbind(
  decision,
  as.data.frame(fp_X),
  log_game_time = log_game_time,
  log_qtr_time = log_qtr_time,
  q2_flag      = q2_flag,
  q4_flag      = q4_flag,
  q2_log_time  = q2_flag * log_qtr_time,
  q4_log_time  = q4_flag * log_qtr_time
)

X$season_num <- as.integer(X$season)
X$action <- factor(X$action, levels = c('go_for_it', 'punt', 'FG'))

# Centered linear season trend (reviewer Main-3: coaching decisions have shifted
# over time, so the propensity model must carry a season term).
# nnet::multinom is fixed-effects only, and each rolling fold trains on just 3
# seasons, so a spline or random effect on season is not identifiable per fold.
# A linear trend centered on the training window costs 1 parameter per class and
# captures within-window drift. The value is filled per fold inside the rolling
# loop below (centering uses the TRAINING window mean, applied to the test
# season as well); initialised here so the term survives the names(X) filter.
X$season_c <- 0

# Model formula
fp_terms  <- paste0('fp_',  seq_len(ncol(fp_X)))
base_terms <- c(
  fp_terms, 'log_game_time',
  'wind_z', 'temp_z', 'ydstogo', 'leverage_z', 'season_c',
  'indoors', 'is_turf', 'high_altitude',
  'is_ot',
  'go_ahead', 'one_score_down', 'to_tie', 'extend_lead', 'two_score_down',
  'q2_log_time', 'q4_log_time'
)
# Every term must be present. Previously this silently dropped missing columns,
# so a renamed or typo'd covariate would vanish from the model without an error.
missing_terms <- setdiff(base_terms, names(X))
if (length(missing_terms)) stop('Missing model terms: ', paste(missing_terms, collapse = ', '))
form_multinom <- as.formula(paste('action ~', paste(base_terms, collapse = ' + ')))

message('Feature matrix built: ', nrow(X), ' rows, ', ncol(X), ' cols')
message('Formula terms: ', length(base_terms), ' | field position basis: ',
        ncol(fp_X), ' cols (ns, knots ', paste(FP_KNOTS, collapse = '/'),
        ', boundary ', paste(FP_BOUNDS, collapse = '/'), ')')
# Guard: no data-derived basis may be built outside the rolling-origin loop.
stopifnot(!any(grepl('^nst_', names(X))))

Feature matrix built: 105249 rows, 104 cols



Formula terms: 23 | field position basis: 6 cols (ns, knots 15/30/45/60/75, boundary 1/100)



## 6. Rolling-Origin OOF Multinomial Fit

For each season s ≥ 2015, train on the prior 3 seasons, predict on season s.
This prevents data leakage and captures era drift in coaching behavior.

In [7]:
SEASONS_ALL <- sort(unique(X$season_num))
SEASONS_OOF <- SEASONS_ALL[SEASONS_ALL >= OOF_START]

all_preds    <- list()
metrics_rows <- list()

for (s in SEASONS_OOF) {
  train_seasons <- seq(max(min(SEASONS_ALL), s - WINDOW), s - 1)
  train_idx <- which(X$season_num %in% train_seasons)
  test_idx  <- which(X$season_num == s)
  if (length(train_idx) < 100 || length(test_idx) == 0) next

  train <- X[train_idx, , drop = FALSE]
  test  <- X[test_idx,  , drop = FALSE]

  # Center the season trend on the TRAINING window mean, and apply the same
  # centering constant to the test season. The test fold therefore sits one
  # step beyond the training centre (season_c = +2 for a 3-season window),
  # which is exactly the one-season-ahead drift extrapolation we want.
  season_center   <- mean(train$season_num, na.rm = TRUE)
  train$season_c  <- train$season_num - season_center
  test$season_c   <- test$season_num  - season_center

  m_multi <- tryCatch(
    nnet::multinom(form_multinom, data = train, trace = FALSE),
    error = function(e) { message('Season ', s, ' failed: ', e$message); NULL }
  )

  p_multi <- rep(NA_real_, nrow(test))
  act_pred <- rep(NA_character_, nrow(test))

  if (!is.null(m_multi)) {
    preds_mat <- tryCatch(predict(m_multi, newdata = test, type = 'probs'), error = function(e) NULL)
    if (!is.null(preds_mat) && 'FG' %in% colnames(preds_mat)) {
      p_multi  <- as.numeric(preds_mat[, 'FG'])
      act_pred <- apply(preds_mat, 1, function(r) colnames(preds_mat)[which.max(r)])
    }
  }

  y_bin       <- test$attempt_fg
  in_range_tr <- train$yardline_100 <= IN_RANGE_YL
  in_range_te <- test$yardline_100  <= IN_RANGE_YL

  # Target prevalence is computed over the KICKABLE subset, not the full field.
  # The decision model is fitted across all field positions because that is the
  # choice a coach actually faces, but M3 reweights observed attempts toward the
  # universe of *plausible* kicks. Using the full-field prevalence here would
  # make M3 target a population containing 100-yard field goals.
  target_prev <- mean(train$attempt_fg[in_range_tr], na.rm = TRUE)

  # Clip to the documented [CLIP_MIN, CLIP_MAX] bounds BEFORE forming weights.
  # This previously clipped at machine epsilon, so the stabilized weights were
  # built on propensities as small as 1e-8 and a single attempt could carry a
  # weight in the thousands. The [0.02, 0.98] clip was applied later, but only
  # to columns the outcome model never used.
  p_clip <- pmin(pmax(p_multi, CLIP_MIN), CLIP_MAX)

  # Stabilized raw IPW
  w_raw <- ifelse(
    y_bin == 1L,
    target_prev / p_clip,
    (1 - target_prev) / (1 - p_clip)
  )

  # Hajek normalize within fold over the in-range subset (mean = 1 there)
  w_hajek <- w_raw / mean(w_raw[in_range_te], na.rm = TRUE)

  fold_preds <- tibble::tibble(
    season               = s,
    season_c             = test$season_c,
    game_id              = test$game_id,
    play_id              = test$play_id,
    attempt_fg           = y_bin,
    yardline_100         = test$yardline_100,
    in_range             = as.integer(in_range_te),
    action_actual        = as.character(test$action),
    action_pred          = as.character(act_pred),
    p_hat_multinom       = p_multi,
    p_hat_attempt_clipped = p_clip,
    p_target_multinom    = target_prev,
    weight_multinom_raw  = w_raw,
    weight_multinom_hajek = w_hajek
  )
  all_preds[[as.character(s)]] <- fold_preds

  auc_val <- tryCatch(as.numeric(pROC::auc(y_bin, p_multi, quiet = TRUE)), error = function(e) NA_real_)

  # AUC on the kickable subset as well. Across the full field the task is close
  # to trivial - a punt from your own 12 is never a field goal - so the full-field
  # AUC is not comparable to the in-range figure the selection argument rests on.
  # Both are reported so the RQ1 claim is read against the right denominator.
  auc_in <- tryCatch(
    as.numeric(pROC::auc(y_bin[in_range_te], p_multi[in_range_te], quiet = TRUE)),
    error = function(e) NA_real_)

  metrics_rows[[paste0('s', s)]] <- tibble::tibble(
    season = s, model = 'multinom',
    auc = auc_val, brier = brier(y_bin, p_multi), logloss = logloss(y_bin, p_multi),
    auc_in_range = auc_in,
    brier_in_range = brier(y_bin[in_range_te], p_multi[in_range_te]),
    n_train = length(train_idx), n_test = length(test_idx),
    n_test_in_range = sum(in_range_te, na.rm = TRUE),
    target_prev = target_prev,
    ess_hajek = ess(w_hajek[y_bin == 1L])
  )

  message(sprintf('Season %d: AUC=%.3f (in-range %.3f) | Brier=%.3f | train_n=%d | test_n=%d',
    s, auc_val, auc_in, brier(y_bin, p_multi), length(train_idx), length(test_idx)))
}

preds_oof   <- dplyr::bind_rows(all_preds) %>%
  mutate(game_id = as.character(game_id), play_id = as.character(play_id))
metrics_oof <- dplyr::bind_rows(metrics_rows)

message('\nOOF predictions: ', nrow(preds_oof), ' rows')
print(metrics_oof)

Season 2015: AUC=0.987 (in-range 0.959) | Brier=0.037 | train_n=12199 | test_n=4087



Season 2016: AUC=0.986 (in-range 0.953) | Brier=0.040 | train_n=12198 | test_n=3975



Season 2017: AUC=0.986 (in-range 0.956) | Brier=0.039 | train_n=12037 | test_n=4117



Season 2018: AUC=0.984 (in-range 0.954) | Brier=0.042 | train_n=12179 | test_n=3874



Season 2019: AUC=0.983 (in-range 0.950) | Brier=0.045 | train_n=11966 | test_n=3883



...

Season 2023: AUC=0.980 (in-range 0.942) | Brier=0.049 | train_n=11953 | test_n=4292



Season 2024: AUC=0.977 (in-range 0.937) | Brier=0.054 | train_n=12550 | test_n=4101



Season 2025: AUC=0.977 (in-range 0.939) | Brier=0.056 | train_n=12562 | test_n=4113




OOF predictions: 44395 rows



# A tibble: 11 × 12
   season model    auc  brier logloss auc_in_range brier_in_range n_train n_test
    <int> <chr>  <dbl>  <dbl>   <dbl>        <dbl>          <dbl>   <int>  <int>
 1   2015 multi… 0.987 0.0365   0.121        0.959         0.0750   12199   4087
 2   2016 multi… 0.986 0.0398   0.130        0.953         0.0810   12198   3975
 3   2017 multi… 0.986 0.0387   0.128        0.956         0.0794   12037   4117
 4   2018 multi… 0.984 0.0419   0.137        0.954         0.0829   12179   3874
 5   2019 multi… 0.983 0.0449   0.145        0.950         0.0881   11966   3883
 6   2020 multi… 0.979 0.0520   0.167        0.940         0.0974   11874   3695
 7   2021 multi… 0.976 0.0555   0.172        0.933         0.104    11452   4089
 8   2022 multi… 0.978 0.0527   0.166        0.935         0.102    11667   4169
 9   2023 multi… 0.980 0.0487   0.154        0.942         0.0955   11953   4292
10   2024 multi… 0.977 0.0542   0.176        0.937         0.0998   12550   4101
11   202

## 7. Full IPW Weight Pipeline

Applied to all rows in the OOF predictions dataframe:
1. Clip to [0.02, 0.98]
2. Stabilized IPW
3. Hájek season-normalize
4. 3σ cap
5. Re-normalize (mean = 1)

In [8]:
# ============================================================
# 7. Full IPW Weight Pipeline
# ============================================================
# One chain, one canonical output. `w_ipw_final` is the weight notebook 03 fits
# M3 with, and it is the weight described in the paper: clipped, stabilized,
# Hajek-normalized over the kickable subset, then 3-sigma capped.
#
# Previously three weight columns were produced and the outcome model was fitted
# with `weight_multinom_hajek` (uncapped, epsilon-clipped, max ~1633) while the
# capped column `w_ipw_final` was used only in M3's row filter. The dead
# intermediates are no longer emitted so that mismatch cannot recur.

# Step 1: Clip propensities to the documented bounds
preds_oof <- preds_oof %>%
  mutate(
    p_hat_attempt_clipped = pmin(pmax(p_hat_multinom, CLIP_MIN), CLIP_MAX),
    # Step 2: Stabilized IPW
    w_ipw_raw = ifelse(
      attempt_fg == 1L,
      p_target_multinom / p_hat_attempt_clipped,
      (1 - p_target_multinom) / (1 - p_hat_attempt_clipped)
    )
  )

# Step 3: Hajek normalize within season, over the in-range subset
preds_oof <- preds_oof %>%
  group_by(season) %>%
  mutate(w_clip_hajek = w_ipw_raw / mean(w_ipw_raw[in_range == 1L], na.rm = TRUE)) %>%
  ungroup()

# Steps 4 & 5: 3-sigma cap then re-normalize over the same in-range subset
preds_oof <- preds_oof %>%
  group_by(season) %>%
  mutate(
    w_mean = mean(w_clip_hajek[in_range == 1L], na.rm = TRUE),
    w_sd   = sd(w_clip_hajek[in_range == 1L],   na.rm = TRUE),
    w_cap  = w_mean + 3 * w_sd,
    w_capped = pmin(w_clip_hajek, w_cap),
    w_ipw_final = w_capped / mean(w_capped[in_range == 1L], na.rm = TRUE)
  ) %>%
  ungroup() %>%
  select(-w_mean, -w_sd, -w_cap, -w_capped, -w_clip_hajek, -w_ipw_raw)

# Guard: the canonical weight must be well behaved on the rows M3 actually fits.
w_fg <- preds_oof$w_ipw_final[preds_oof$attempt_fg == 1L]
stopifnot(all(is.finite(w_fg)), all(w_fg > 0))
message(sprintf('w_ipw_final on FG attempts: min=%.3f median=%.3f mean=%.3f max=%.3f',
                min(w_fg), median(w_fg), mean(w_fg), max(w_fg)))
if (max(w_fg) > 50) warning('w_ipw_final max exceeds 50 - check the 3-sigma cap')


w_ipw_final on FG attempts: min=0.628 median=0.713 mean=0.954 max=8.493



In [9]:
# ============================================================
# 8. Weight Diagnostics
# ============================================================
diag <- preds_oof %>%
  filter(attempt_fg == 1L) %>%
  summarise(
    n         = n(),
    min_w     = min(w_ipw_final, na.rm = TRUE),
    median_w  = median(w_ipw_final, na.rm = TRUE),
    mean_w    = mean(w_ipw_final, na.rm = TRUE),
    p99_w     = quantile(w_ipw_final, 0.99, na.rm = TRUE),
    max_w     = max(w_ipw_final, na.rm = TRUE),
    sd_w      = sd(w_ipw_final, na.rm = TRUE),
    ess       = ess(w_ipw_final),
    ess_ratio = ess / n
  )
cat('\n=== FG Attempt IPW Weight Summary ===\n')
print(diag)

diag_by_season <- preds_oof %>%
  filter(attempt_fg == 1L) %>%
  group_by(season) %>%
  summarise(
    n = n(), mean_w = mean(w_ipw_final, na.rm = TRUE),
    max_w = max(w_ipw_final, na.rm = TRUE), ess = ess(w_ipw_final),
    .groups = 'drop'
  )
print(diag_by_season)


=== FG Attempt IPW Weight Summary ===


# A tibble: 1 × 9
      n min_w median_w mean_w p99_w max_w  sd_w   ess ess_ratio
  <int> <dbl>    <dbl>  <dbl> <dbl> <dbl> <dbl> <dbl>     <dbl>
1 11764 0.628    0.713  0.954  6.68  8.49 0.882 6345.     0.539


# A tibble: 11 × 5
   season     n mean_w max_w   ess
    <int> <int>  <dbl> <dbl> <dbl>
 1   2015  1033  0.929  7.96  591.
 2   2016  1050  0.876  7.53  610.
 3   2017  1065  0.937  8.49  534.
 4   2018   989  0.938  7.51  543.
 5   2019  1018  0.949  7.47  602.
 6   2020  1015  1.02   7.17  527.
 7   2021  1076  0.919  6.87  659.
 8   2022  1105  0.997  7.45  611.
 9   2023  1107  0.980  6.31  676.
10   2024  1166  0.943  8.46  566.
11   2025  1140  0.999  8.36  517.


In [10]:
# ============================================================
# 9. Save Outputs
# ============================================================
out_preds   <- file.path(reports_pi_dir, 'attempt_pi_oof_predictions_final.csv')
out_metrics <- file.path(reports_pi_dir, 'attempt_pi_metrics_by_season.csv')

readr::write_csv(preds_oof,   out_preds)
readr::write_csv(metrics_oof, out_metrics)

message('\n=== Outputs Written ===')
message('OOF predictions: ', out_preds, ' (', nrow(preds_oof), ' rows)')
message('Season metrics:  ', out_metrics)


=== Outputs Written ===



OOF predictions: X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/attempt_pi/attempt_pi_oof_predictions_final.csv (44395 rows)



Season metrics:  X:/My Files/Sports Analytics Projects/Football/Kickers/NFL-Field-Goal-Kicker-Model/reports/attempt_pi/attempt_pi_metrics_by_season.csv

